# Delaunay Triangulation

In this exercise, we will do Delaunay triangulation by means of the the edge-flip algorithm.


In [ ]:
from pygel3d import hmesh, spatial, jupyter_display as jd
import numpy as np
from queue import Queue
import plotly.offline as py
import plotly.graph_objs as go
array = np.array
jd.set_export_mode(True)

### Delaunay edge

The following function needs to check whether a given half-edge fullfills the Delaunay property. This can be done by means of the in-circle predicate. In practie, we need to build a 4x4 matrix and check its determinant.

In [ ]:
def delaunay_edge(m,h):
    
    if m.is_halfedge_at_boundary(h):
        return True

    pos = m.positions()
    A = m.incident_vertex(m.prev_halfedge(h))
    B = m.incident_vertex(h)
    C = m.incident_vertex(m.next_halfedge(h))
    D = m.incident_vertex(m.next_halfedge(m.opposite_halfedge(h)))

    ax, ay, az = pos[A]
    bx, by, bz = pos[B]
    cx, cy, cz = pos[C]
    dx, dy, dz = pos[D]

    M = np.array([[ax, ay, ax**2+ay**2, 1],
                [bx, by, bx**2+by**2, 1],
                [cx, cy, cx**2+cy**2, 1],
                [dx, dy, dx**2+dy**2, 1]])

    det_M = np.linalg.det(M)

    if det_M < 0:
        return True
        
    # Below compute and return the value of the the in-circle predicate.


### Test function
Code below tests the `delaunay_edge` function

In [ ]:
m_test = hmesh.Manifold()
m_test.add_face([[0.0,0.0,0.0],[1.0,0.0,0.0],[1.0,1.0,0.0]])
m_test.add_face([[0.0,0.0,0.0],[1.0,1.0,0.0],[0.45,0.55,0.0]])
hmesh.stitch(m_test)
jd.display(m_test)
for h in m_test.halfedges():
    if not delaunay_edge(m_test, h):
        m_test.flip_edge(h)
        print("flipped")
    print("no flip")
jd.display(m_test)

### Barycentric coordinates

For the following, we need to determine the barycentric coordinates of a point on a given face. In the following function 

``m`` is a manifold

``f`` a given face

``p`` are the point coordinates

In [ ]:
def barycentrics(m, f, p):
    pos = m.positions()
    y = p[:2]
    vertices = m.circulate_face(f, mode='v')

    x0 = pos[vertices[0]][:2]
    x1 = pos[vertices[1]][:2]
    x2 = pos[vertices[2]][:2]

    A0 = np.linalg.det(np.array([x1-y, x2-y]))
    A1 = np.linalg.det(np.array([x2-y, x0-y]))
    A2 = np.linalg.det(np.array([x0-y, x1-y]))

    total = A0 + A1 + A2

    a0 = A0 / total
    a1 = A1 / total
    a2 = A2 / total

    return a0, a1, a2

In [ ]:
print("b0 = ", barycentrics(m_test,0,[0.4,0.4,0]))
print("b1 = ", barycentrics(m_test,1,[0.4,0.4,0]))

### Filtering points

The following function reduces the number of poins.

``coords`` ar the coordinates of points to be filtered

``rad`` is a radius

In [ ]:
def create_2d_tree(pts):
    tree = spatial.I3DTree()
    for i in range(len(pts)):
        p = array(pts[i])
        p[2] = 0
        tree.insert(p,i)
    tree.build()
    return tree

def filter_points(coords, rad):
    tree = create_2d_tree(coords)
    new_coords = []
    visited = [False]*len(coords)
    for i in range(0,len(coords)):
        if not visited[i]:
            p = array(coords[i])
            new_coords += [array(p)]
            p[2] = 0
            (K,V) = tree.in_sphere(p,rad)
            for idx in V:
                visited[idx] = True
    return new_coords

## Delaunay triangulation pipeline
Now we are ready to run a complete Delayanay triangulation pipeline. 

### Importing data
The code below imports 3D point data and normalizes coordinates such that the x and y components lie within a unit square.

In [ ]:
f = open("./kote1-sorted.txt")
lines = f.readlines()
coords = []
for l in lines:
    coords += [list(map(float, l.split()))]
point_mat = np.array(coords)
spanx = point_mat[:,0].max() - point_mat[:,0].min()
spany = point_mat[:,1].max() - point_mat[:,1].min()
span = max(spanx,spany)
coords -= np.array([point_mat[:,0].min(),point_mat[:,1].min(),point_mat[:,2].min()])
coords *= np.array([1.,1.,5.0])/span

coords = filter_points(coords, 0.001)

### Adding points
You should now add the points in ``coords`` to the manifold ``m`` one by one.

In [ ]:
m = hmesh.Manifold()

# One big "helper triangle" to encompass all normalized points
m.add_face([[-1.,0.,0.],[1.,0.,0.],[1.,2.,0.]])

for c in coords:
    print("Inserting point : ", c)
    # Below, insert algorithmic part of Delaunay triangulation.
    # The code should:
    # - find the appropriate triangle containing c and split it, inserting c
    # - Flip all edges which become not locally Delaunay in the process.
    for face in m.faces():
        a0, a1, a2 = barycentrics(m,face,c)
        if a0 >= 0 and a1 >= 0 and a2 >= 0:
            v = m.split_face_by_vertex(face)
            m.positions()[v] = c
            break

    q = Queue()
    for h in m.circulate_vertex(v, mode='h'):
        q.put(m.next_halfedge(h))

    while not q.empty():
        h = q.get()
        if not m.halfedge_in_use(h):
            continue
        if not delaunay_edge(m,h):
            h = m.flip_edge(h)
            for h2 in m.circulate_vertex(v, mode='h'):
                q.put(m.next_halfedge(h2))

       
# Removing the "helper triangle"
m.remove_vertex(0)
m.remove_vertex(1)
m.remove_vertex(2)

# Removing caps at edge of mesh
for iter in range(15):
    avg_len = hmesh.average_edge_length(m)
    for h in m.halfedges():
        if m.is_halfedge_at_boundary(h) and m.halfedge_in_use(h):
            if m.edge_length(h)>2.5 * avg_len:
                m.remove_edge(h)
m.cleanup()

print("Checking that all edges are Delaunay...")
we_are_good = True
for h in m.halfedges():
    if not delaunay_edge(m,h):
        print("Found a non-Delaunay edge!")
        we_are_good = False
if we_are_good:
    print("We are good!")

In [ ]:
jd.display(m)

### Questions

- Discuss the difference between the notions of locally Delaunay and globally Delaunay
- What is the most efficient Delaunay triangulation algorithm?
- What is most important to the efficiency of this Delaunay triangulation algorithm? \
MINE
- Why is it a problem if four points share a common circumcircle? What would this algorithm do? \
MINE